# Тестирование градиентного бустинга.

In [24]:
import polars as pl
import math
import sys
import os
import numpy as np
import math
import mlflow
import mlflow.catboost
import boto3
from catboost import CatBoostRanker, Pool
import optuna
from dotenv import load_dotenv
import gc

In [2]:
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.dataset import global_temporal_split, ndcg_at_k, precision_recall_at_k, create_target, PolarsFrame

2026-05-26 14:35:27.675 | INFO     | src.config:<module>:12 - PROJ_ROOT path is: /home/alex/VS_python_projects/DreamTeamRecSys


In [3]:
events_path = "../data/dataset/small/marketplace/events"
items_path = "../data/dataset/small/marketplace/items.pq"

In [4]:
events_df = pl.scan_parquet(events_path)
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String)])

In [5]:
events_df = create_target(events_df, target_type="log_target")
events_df = events_df.drop("len")

In [7]:
null_info = events_df.null_count().collect()
print(null_info)

shape: (1, 7)
┌───────────┬─────────┬─────────┬───────────┬─────────────┬─────┬────────┐
│ timestamp ┆ user_id ┆ item_id ┆ subdomain ┆ action_type ┆ os  ┆ target │
│ ---       ┆ ---     ┆ ---     ┆ ---       ┆ ---         ┆ --- ┆ ---    │
│ u32       ┆ u32     ┆ u32     ┆ u32       ┆ u32         ┆ u32 ┆ u32    │
╞═══════════╪═════════╪═════════╪═══════════╪═════════════╪═════╪════════╡
│ 0         ┆ 0       ┆ 0       ┆ 41792     ┆ 0           ┆ 0   ┆ 0      │
└───────────┴─────────┴─────────┴───────────┴─────────────┴─────┴────────┘


Посмотрю по таблице items, какие есть пропуски.

In [6]:
ivents_lf = pl.scan_parquet(items_path)
ivents_lf.head().collect(engine="streaming")

item_id,brand_id,category,subcategory,price,embedding
str,u64,str,str,f64,"array[f32, 300]"
"""nfmcg_10000001""",137356,null,null,6.06364,"[-0.070853, 0.03246, … 0.082178]"
"""nfmcg_10000012""",53389,null,null,0.960436,"[-0.091099, 0.043168, … 0.060287]"
"""nfmcg_1000002""",34153,"""Fashion Accessories, Tech Add-…","""Hats, Scarves, and Shawls""",-1.793113,"[-0.056533, 0.082878, … 0.037475]"
"""nfmcg_10000034""",39543,null,null,3.416755,"[-0.112588, 0.043333, … -0.001615]"
"""nfmcg_10000039""",82320,"""Fashion Accessories, Tech Add-…","""Jewelry and Costume Jewelry""",4.293433,"[-0.157201, 0.026234, … 0.013904]"


In [7]:
null_info = ivents_lf.null_count().collect()
print(null_info)

shape: (1, 6)
┌─────────┬──────────┬──────────┬─────────────┬───────┬───────────┐
│ item_id ┆ brand_id ┆ category ┆ subcategory ┆ price ┆ embedding │
│ ---     ┆ ---      ┆ ---      ┆ ---         ┆ ---   ┆ ---       │
│ u32     ┆ u32      ┆ u32      ┆ u32         ┆ u32   ┆ u32       │
╞═════════╪══════════╪══════════╪═════════════╪═══════╪═══════════╡
│ 0       ┆ 0        ┆ 966395   ┆ 1233023     ┆ 2882  ┆ 73        │
└─────────┴──────────┴──────────┴─────────────┴───────┴───────────┘


Подготовлю данные для дальнейшей работы с моделями. Категориальные признаки оставляю, так как бустинги умеют с ними работать. Удаляю товары с пропусками в цене, так как их минимальное количество от общего числа. Эмбеддинги пока не беру, пропуски в категориях и подкатегориях заменяю на новую категорию "unknown". Строки в таблице действий с пропусками в поддомене также просто удаляю, так как их не много. Далее форматирую временную отметку в таблице событий и собираю все в одну таблицу events_prepared, с которой и продолжу работу.

In [8]:
ACCEPTABLE_ACTIONS = ["view", "click", "clickout", "like"]

# Очистка items
items_lf = (
    ivents_lf
    .drop_nulls(subset=["price"])
    .with_columns([
        pl.col("category").fill_null("unknown").alias("category"),
        pl.col("subcategory").fill_null("unknown").alias("subcategory"),
        pl.col("brand_id").fill_null(-1).alias("brand_id"),
    ])
    .select([
        "item_id",
        "brand_id",
        "category",
        "subcategory",
        "price",
    ])
)

# Подготовка events
MICROS_IN_DAY = 24 * 60 * 60 * 1_000_000

events_prepared = (
    events_df
    .filter(pl.col("action_type").is_in(ACCEPTABLE_ACTIONS))
    # Удаляем старый join с actions_count, так как create_target уже сделал его!
    # .join(actions_count.lazy(), on="action_type", how="inner") 
    .join(items_lf, on="item_id", how="inner")
    .with_columns([
        (pl.col("timestamp").cast(pl.Int64) // MICROS_IN_DAY).alias("day"),
    ])
    .filter(pl.col("subdomain").is_not_null())
    .select([
        "timestamp",
        "day",
        "user_id",
        "item_id",
        "subdomain",
        "action_type",
        "os",
        "target",
        "brand_id",
        "category",
        "subcategory",
        "price",
    ])
)

In [11]:
events_prepared.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('day', Int64),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('target', Float64),
        ('brand_id', Int64),
        ('category', String),
        ('subcategory', String),
        ('price', Float64)])

In [12]:
events_prepared.head().collect(engine="streaming")

timestamp,day,user_id,item_id,subdomain,action_type,os,target,brand_id,category,subcategory,price
duration[μs],i64,u64,str,str,str,str,f64,i64,str,str,f64
1082d 18h 41m 46s 715560µs,1082,43228442,"""nfmcg_21388948""","""u2i""","""click""","""android""",3.571844,111440,"""Fashion Accessories, Tech Add-…","""Jewelry and Costume Jewelry""",1.070478
1082d 18h 42m 58s 271056µs,1082,24154342,"""nfmcg_12746270""","""u2i""","""click""","""android""",3.571844,137356,"""unknown""","""unknown""",2.436604
1082d 18h 43m 57s 469191µs,1082,78531712,"""nfmcg_9528279""","""u2i""","""click""","""android""",3.571844,117885,"""unknown""","""unknown""",2.69984
1082d 18h 45m 31s 645289µs,1082,58302299,"""nfmcg_8987827""","""other""","""click""","""ios""",3.571844,166340,"""Fashion Accessories, Tech Add-…","""Jewelry and Costume Jewelry""",0.117105
1082d 18h 47m 2s 499295µs,1082,81373793,"""nfmcg_3347544""","""search""","""click""","""ios""",3.571844,95522,"""unknown""","""unknown""",0.94928


Беру данные для обучения за 5% последних дней - максимум, который позволяет моя оперативная память (32гб).

In [9]:
#границы дней
raw_events = pl.scan_parquet(events_path)
MICROS_IN_DAY = 24 * 60 * 60 * 1_000_000

min_day, max_day = (
    raw_events
    .select(
        (pl.col("timestamp").cast(pl.Int64) // MICROS_IN_DAY).min().alias("min_day"),
        (pl.col("timestamp").cast(pl.Int64) // MICROS_IN_DAY).max().alias("max_day"),
    )
    .collect()
    .row(0)
)

days_all = (max_day - min_day) + 1
n_days_subset = int(days_all * 0.05)
if n_days_subset < 1:
    n_days_subset = 1

subset_min_day = max_day - n_days_subset + 1

events_subset = events_prepared.filter(pl.col("day") >= subset_min_day)

In [10]:
train, test = global_temporal_split(events_subset, 0.2)
print("Количество строк train:", train.select(pl.len()).collect().item())
print("Количество колонок train:", len(train.columns))
print("Количество строк test:", test.select(pl.len()).collect().item())
print("Количество колонок test:", len(test.columns))

Количество строк train: 4044137
Количество колонок train: 12


/tmp/ipykernel_97668/3991756773.py:3: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print("Количество колонок train:", len(train.columns))


Количество строк test: 1072961
Количество колонок test: 12


/tmp/ipykernel_97668/3991756773.py:5: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print("Количество колонок test:", len(test.columns))


In [11]:
# агрегируем события до уровня user-item в train и test
train_ui = (
    train
    .group_by(["user_id", "item_id"])
    .agg([
        pl.col("target").max().alias("target"),
        pl.col("day").max().alias("last_day"),       
        pl.col("price").mean().alias("price_mean"),
        pl.col("category").first().alias("category"),
        pl.col("subcategory").first().alias("subcategory"),
        pl.col("brand_id").first().alias("brand_id"),
        pl.col("os").first().alias("os"),
        pl.col("subdomain").first().alias("subdomain"),
        pl.col("action_type").first().alias("last_action_type"),
    ])
)

test_ui = (
    test
    .group_by(["user_id", "item_id"])
    .agg([
        pl.col("target").max().alias("target"),
        pl.col("day").max().alias("last_day"),
        pl.col("price").mean().alias("price_mean"),
        pl.col("category").first().alias("category"),
        pl.col("subcategory").first().alias("subcategory"),
        pl.col("brand_id").first().alias("brand_id"),
        pl.col("os").first().alias("os"),
        pl.col("subdomain").first().alias("subdomain"),
        pl.col("action_type").first().alias("last_action_type"),
    ])
)

In [12]:
train_ui.head().collect(engine="streaming")

user_id,item_id,target,last_day,price_mean,category,subcategory,brand_id,os,subdomain,last_action_type
u64,str,f64,i64,f64,str,str,i64,str,str,str
47805966,"""nfmcg_5877383""",0.710044,1299,4.316762,"""Miscellaneous Goods (Uncategor…","""unknown""",137311,"""android""","""u2i""","""view"""
80577284,"""nfmcg_16254503""",0.710044,1303,-0.616532,"""unknown""","""unknown""",141256,"""android""","""other""","""view"""
66235655,"""nfmcg_27233617""",0.710044,1299,-1.985188,"""Cosmetics, Personal Care, and …","""Body Skin Hygiene and Care Pro…",52917,"""ios""","""search""","""view"""
83549513,"""nfmcg_15867674""",0.710044,1299,4.811988,"""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …",117885,"""android""","""i2i""","""view"""
74702898,"""nfmcg_14156510""",0.710044,1299,3.457901,"""Electronic Devices and Gadgets""","""Computer Hardware Components""",38547,"""android""","""i2i""","""view"""


In [13]:
test_ui.head().collect(engine="streaming")

user_id,item_id,target,last_day,price_mean,category,subcategory,brand_id,os,subdomain,last_action_type
u64,str,f64,i64,f64,str,str,i64,str,str,str
7636163,"""nfmcg_17237351""",0.710044,1308,2.234918,"""unknown""","""unknown""",12499,"""other""","""catalog""","""view"""
75537319,"""nfmcg_22415508""",0.710044,1308,2.77053,"""Household Electrical Appliance…","""Large Kitchen Appliances""",233267,"""android""","""catalog""","""view"""
28184195,"""nfmcg_18200186""",0.710044,1307,-0.631637,"""unknown""","""unknown""",141256,"""android""","""other""","""view"""
16988341,"""nfmcg_16591964""",0.710044,1308,3.611491,"""unknown""","""unknown""",12499,"""ios""","""catalog""","""view"""
39990139,"""nfmcg_18200186""",0.710044,1307,-0.631637,"""unknown""","""unknown""",141256,"""android""","""other""","""view"""


Заведу еще валидационную выборку.

In [14]:
min_day, max_day = (
    train_ui.select(
        pl.col("last_day").min().alias("min_day"),
        pl.col("last_day").max().alias("max_day"),
    ).collect().row(0)
)

days_all = (max_day - min_day) + 1
val_days = max(int(days_all * 0.1), 1)
val_min_day = max_day - val_days + 1

train_train_lf = train_ui.filter(pl.col("last_day") < val_min_day)
valid_lf = train_ui.filter(pl.col("last_day") >= val_min_day)

Добавлю новые признаки - агрегаты по пользователям, товарам и истории взаимодействия user-item. Эти признаки позже передам для обучения модели.

In [15]:
# user/popularity агрегаты на train_train
user_aggs = (
    train_train_lf
    .group_by("user_id")
    .agg([
        pl.len().alias("user_hist_len"),
        pl.col("target").mean().alias("user_mean_target"),
    ])
)

item_aggs = (
    train_train_lf
    .group_by("item_id")
    .agg([
        pl.len().alias("item_popularity"),
        pl.col("price_mean").mean().alias("price_mean_item"),
    ])
)

In [16]:
print(user_aggs.head().collect(engine="streaming"))
print("________________________")
print(item_aggs.head().collect(engine="streaming"))

shape: (5, 3)
┌──────────┬───────────────┬──────────────────┐
│ user_id  ┆ user_hist_len ┆ user_mean_target │
│ ---      ┆ ---           ┆ ---              │
│ u64      ┆ u32           ┆ f64              │
╞══════════╪═══════════════╪══════════════════╡
│ 29691022 ┆ 89            ┆ 0.710044         │
│ 31113618 ┆ 12            ┆ 1.187011         │
│ 18380534 ┆ 686           ┆ 0.935832         │
│ 52641148 ┆ 34            ┆ 0.710044         │
│ 27435332 ┆ 16            ┆ 0.710044         │
└──────────┴───────────────┴──────────────────┘
________________________
shape: (5, 3)
┌────────────────┬─────────────────┬─────────────────┐
│ item_id        ┆ item_popularity ┆ price_mean_item │
│ ---            ┆ ---             ┆ ---             │
│ str            ┆ u32             ┆ f64             │
╞════════════════╪═════════════════╪═════════════════╡
│ nfmcg_2648313  ┆ 147             ┆ 4.06907         │
│ nfmcg_6124054  ┆ 79              ┆ -0.223129       │
│ nfmcg_906223   ┆ 1              

In [17]:
def prep_rank_lf(base_lf: pl.LazyFrame) -> pl.LazyFrame:
    return (
        base_lf
        .join(user_aggs, on="user_id", how="left")
        .join(item_aggs, on="item_id", how="left")
    )

train_rank_lf = prep_rank_lf(train_train_lf)
valid_rank_lf = prep_rank_lf(valid_lf)

train_df = train_rank_lf.collect().to_pandas(use_pyarrow_extension_array=False)
valid_df = valid_rank_lf.collect().to_pandas(use_pyarrow_extension_array=False)

In [18]:
feature_cols = [
    "item_id",
    "price_mean",
    "price_mean_item",
    "category",
    "subcategory",
    "brand_id",
    "user_hist_len",
    "user_mean_target",
    "item_popularity",
    "os",
    "subdomain",
    "last_day",
]

cat_features = ["item_id", "category", "subcategory", "brand_id", "os", "subdomain"]

In [19]:
train_df = train_df.sort_values(["user_id", "item_id"])
valid_df = valid_df.sort_values(["user_id", "item_id"])

Обучаю первую модель- вариант CatBoost для задач ранжирования CatBoostRanker с дефолтной функцией потерь для ранжирования YetiRank, беру ее так как она хорошо работает с категориальными признаками и не требует тонких настроек гиперпараметров, можно сразу увидеть результат для сравнения с нашим бейзлайном - SVD. 

In [20]:
train_pool = Pool(
    data=train_df[feature_cols],
    label=train_df["target"],
    group_id=train_df["user_id"],
    cat_features=cat_features,
)

valid_pool = Pool(
    data=valid_df[feature_cols],
    label=valid_df["target"],
    group_id=valid_df["user_id"],
    cat_features=cat_features,
)

Реализую функцию вычисления по батчам с возможностью выбрать размер батча и top k товаров.

In [21]:
def get_topk_from_toppop(
    model,
    feature_cols,
    cat_features,
    user_test_truth: pl.LazyFrame,
    top_items_lf: pl.LazyFrame,
    batch_size: int = 80, # на 100 иногда вылетает, пробую чуть меньше
    top_k: int = 15,
) -> pl.DataFrame:

    item_features = (
        top_items_lf
        .join(item_aggs, on="item_id", how="left")
        .join(train_ui.select("item_id", "price_mean", "category", "subcategory", "brand_id").unique(subset=["item_id"]), on="item_id", how="left")
    ).collect()

    user_features = (
        user_test_truth.select("user_id").unique()
        .join(user_aggs, on="user_id", how="left")
        .join(train_ui.select("user_id", "os", "subdomain", "last_day").unique(subset=["user_id"]), on="user_id", how="left")
    ).collect()

    user_ids = user_features["user_id"].to_list()
    n_users = len(user_ids)
    n_batches = math.ceil(n_users / batch_size)
    all_rows = []

    for b in range(n_batches):
        s = b * batch_size
        e = min((b + 1) * batch_size, n_users)
        batch_users = user_ids[s:e]
        
        # Берем фичи только нужных юзеров
        batch_user_features = user_features.filter(pl.col("user_id").is_in(batch_users))
        cand_df = batch_user_features.join(item_features, how="cross").to_pandas()
        
        # Заполняем пропуски
        for col in cat_features:
            cand_df[col] = cand_df[col].astype("string").fillna("__nan__")
            
        cand_df["user_hist_len"] = cand_df["user_hist_len"].fillna(0).astype("int32")
        cand_df["item_popularity"] = cand_df["item_popularity"].fillna(0).astype("int32")
        cand_df["user_mean_target"] = cand_df["user_mean_target"].fillna(0.0)
        cand_df["price_mean_item"] = cand_df.get("price_mean_item", cand_df["price_mean"])
        
        # Делаем предикт
        pool = Pool(
            data=cand_df[feature_cols],
            cat_features=cat_features,
        )
        
        cand_df["pred_score"] = model.predict(pool)
        
        # Сортируем и берем top_k
        batch_pl = pl.from_pandas(cand_df[["user_id", "item_id", "pred_score"]])
        batch_topk = (
            batch_pl
            .sort(["user_id", "pred_score"], descending=[False, True])
            .group_by("user_id")
            .agg([
                pl.col("item_id").head(top_k).alias("predicted_items"),
                pl.col("pred_score").head(top_k).alias("predicted_scores"),
            ])
        )
        
        truth_small = user_test_truth.filter(pl.col("user_id").is_in(batch_users)).collect()
        batch_eval = truth_small.join(batch_topk, on="user_id", how="inner")
        all_rows.append(batch_eval)
        
    return pl.concat(all_rows)

Беру только 5000 самых популярных товаров для кандидатогенерации, тк не хватает оперативной памяти. Реализую сохранение эксперимента в mlflow.

In [22]:
TOP_M_ITEMS = 5000
TOP_K = 15

top_items_lf = (
    train_ui
    .group_by("item_id")
    .agg(pl.len().alias("cnt"))
    .sort("cnt", descending=True)
    .limit(TOP_M_ITEMS)
    .select("item_id")
)

user_test_truth = (
    test_ui
    .group_by("user_id")
    .agg([
        pl.col("item_id").alias("true_items"),
        pl.col("target").alias("relevancy"),
    ])
)

In [23]:
load_dotenv("../.env") 

# Переопределяю адрес, так как у меня заняты порты на локальной машине
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://localhost:19000"

mlflow.set_tracking_uri("http://localhost:5050")
experiment_name = "Catboost_Ranker_V2"

try:
    exp_id = mlflow.create_experiment(
        name=experiment_name,
        artifact_location="s3://firstmlflow/mlflow"
    )
except mlflow.exceptions.MlflowException:
    exp_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

mlflow.set_experiment(experiment_name)

print(f"Подключено к MLflow. Эксперимент: {experiment_name}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

Подключено к MLflow. Эксперимент: Catboost_Ranker_V2
Tracking URI: http://localhost:5050


In [ ]:
with mlflow.start_run(run_name="baseline_cb_ranker"):

    params = {
        "loss_function": "YetiRank",
        "eval_metric": f"NDCG:top={TOP_K}",
        "iterations": 300,
        "learning_rate": 0.05,
        "depth": 8,
        "random_seed": 42,
        "verbose": False, # убираю, чтоб немного снизить нагрузку на VS Code, он несколько раз вылетел во время обучения
        "early_stopping_rounds": 50,
        "task_type": "CPU",
    }
    mlflow.log_params(params)
    
    print("Обучение CatBoost...")
    model = CatBoostRanker(**params)
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)
    
    print("Обучение завершено. Считаю метрики...")
    
    topk_df = get_topk_from_toppop(
        model=model,
        feature_cols=feature_cols,
        cat_features=cat_features,
        user_test_truth=user_test_truth,
        top_items_lf=top_items_lf,
        top_k=TOP_K,
    )

    ndcg_df = ndcg_at_k(
        user_based_df=topk_df,
        relevancy_col="relevancy",
        true_items_col="true_items",
        predicted_items_col="predicted_items",
        predicted_score_col="predicted_scores",
        top_k=TOP_K,
    )
    mean_ndcg = ndcg_df["ndcg"].mean()

    precision_k, recall_k = precision_recall_at_k(
        df=topk_df.select(["true_items", "predicted_items"]),
        true_items_col="true_items",
        predicted_items_col="predicted_items",
    )
   

In [ ]:
with mlflow.start_run(run_name="baseline_catboost_v2"):
    mlflow.log_params(params)

    mlflow.log_metrics({
        f"NDCG_{TOP_K}": mean_ndcg,
        f"Precision_{TOP_K}": precision_k,
        f"Recall_{TOP_K}": recall_k
    })
    
    mlflow.catboost.log_model(model, "model")
    
    print("Метрики сохранены в mlflow")

Результаты сохраненились в MLflow и S3 хранилище, процесс обучения и подсчета метрик занял более 30 минут.

Метрики бейзлайн CatBoost


NDCG_15 - 0.1317122572714359

Precision_15 - 0.06665796282669073

Recall_15 - 0.21059404316764524


Далее дополнительно подберу оптимальные гиперпараметры с помощью Optuna. Буду подбирать самые важные гиперпараметры: learning rate, глубину деревьев, коэффициент L2 регуляризации.

In [25]:
with mlflow.start_run(run_name="CatBoost_Optuna_Best") as parent_run:

    def objective(trial):
        with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
            
            params = {
                "loss_function": "YetiRank",
                "eval_metric": f"NDCG:top={TOP_K}",
                "iterations": 300,
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "depth": trial.suggest_int("depth", 4, 10),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
                "random_seed": 42,
                "verbose": 0, # Снижаю нагрузку
                "task_type": "CPU",
                "early_stopping_rounds": 30,
            }
            
            mlflow.log_params(params)

            model = CatBoostRanker(**params)
            model.fit(
                train_pool,
                eval_set=valid_pool,
                use_best_model=True,
                verbose=False,
            )

            # Подсчет метрик
            valid_scored = valid_df.copy()
            valid_scored["pred"] = model.predict(valid_df[feature_cols])
            valid_scored = valid_scored.sort_values(["user_id", "pred"], ascending=[True, False])

            pred_df = (
                pl.from_pandas(valid_scored[["user_id", "item_id", "pred"]])
                .group_by("user_id")
                .agg(
                    pl.col("item_id").head(TOP_K).alias("predicted_items"),
                    pl.col("pred").head(TOP_K).alias("predicted_scores"),
                )
            )

            truth_df = (
                pl.from_pandas(valid_df[["user_id", "item_id", "target"]])
                .group_by("user_id")
                .agg(
                    pl.col("item_id").alias("true_items"),
                    pl.col("target").alias("relevancy"),
                )
            )

            eval_df = truth_df.join(pred_df, on="user_id", how="inner")

            ndcg_df = ndcg_at_k(
                user_based_df=eval_df,
                relevancy_col="relevancy",
                true_items_col="true_items",
                predicted_items_col="predicted_items",
                predicted_score_col="predicted_scores",
                top_k=TOP_K,
            )

            val_ndcg = float(ndcg_df["ndcg"].mean())
            mlflow.log_metric(f"Val_NDCG_{TOP_K}", val_ndcg)
            
            # Удаляю тяжелые объекты из RAM, чтобы VS Code не вылетел
            del model, valid_scored, pred_df, truth_df, eval_df, ndcg_df
            gc.collect()

            return val_ndcg

    # чтобы видеть прогресс без зависаний
    def print_progress(study, trial):
        print(f"Завершен шаг {trial.number}/25 | Лучший NDCG: {study.best_value:.4f}")

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(objective, n_trials=25, timeout=600, callbacks=[print_progress])
    
    print("\nЛучшие параметры найдены:", study.best_params)
    mlflow.log_params(study.best_params)

    print("Обучаем финальную модель...")
    model_best = CatBoostRanker(
        loss_function="YetiRank",
        eval_metric=f"NDCG:top={TOP_K}",
        iterations=300,
        learning_rate=study.best_params['learning_rate'],
        depth=study.best_params['depth'],
        l2_leaf_reg=study.best_params['l2_leaf_reg'],
        random_seed=42,
        verbose=0,
        early_stopping_rounds=50,
        task_type="CPU",
    )

    model_best.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
        verbose=False,
    )

    print("Считаю финальные метрики на тесте...")
    best_topk_df = get_topk_from_toppop(
        model=model_best,
        feature_cols=feature_cols,
        cat_features=cat_features,
        user_test_truth=user_test_truth,
        top_items_lf=top_items_lf,
        top_k=TOP_K,
    )

    ndcg_df_best = ndcg_at_k(
        user_based_df=best_topk_df,
        relevancy_col="relevancy",
        true_items_col="true_items",
        predicted_items_col="predicted_items",
        predicted_score_col="predicted_scores",
        top_k=TOP_K,
    )

    mean_best_ndcg = ndcg_df_best["ndcg"].mean()

    best_precision_k, best_recall_k = precision_recall_at_k(
        df=best_topk_df.select(["true_items", "predicted_items"]),
        true_items_col="true_items",
        predicted_items_col="predicted_items",
    )

    mlflow.log_metrics({
        f"Best_NDCG_{TOP_K}": mean_best_ndcg,
        f"Best_Test_Precision_{TOP_K}": best_precision_k,
        f"Best_Test_Recall_{TOP_K}": best_recall_k
    })
    
    mlflow.catboost.log_model(model_best, "model")

    print("---")
    print(f"Best mean NDCG@{TOP_K}: {mean_best_ndcg:.4f}")
    print(f"Best Precision@{TOP_K}: {best_precision_k:.4f}")
    print(f"Best Recall@{TOP_K}: {best_recall_k:.4f}")

[I 2026-05-26 15:19:05,404] A new study created in memory with name: no-name-a05dc308-1bd1-4422-91c7-91accd4ceea9
[I 2026-05-26 15:24:58,282] Trial 0 finished with value: 0.9813587808725716 and parameters: {'learning_rate': 0.023688639503640783, 'depth': 10, 'l2_leaf_reg': 7.587945476302646}. Best is trial 0 with value: 0.9813587808725716.


🏃 View run trial_0 at: http://localhost:5050/#/experiments/10/runs/b92f642e48e74f8da1675dec83d7087b
🧪 View experiment at: http://localhost:5050/#/experiments/10
Завершен шаг 0/25 | Лучший NDCG: 0.9814


[I 2026-05-26 15:28:57,269] Trial 1 finished with value: 0.981227645290221 and parameters: {'learning_rate': 0.03968793330444373, 'depth': 5, 'l2_leaf_reg': 2.403950683025824}. Best is trial 0 with value: 0.9813587808725716.


🏃 View run trial_1 at: http://localhost:5050/#/experiments/10/runs/fef97a40f97d4143810ace918cef36d8
🧪 View experiment at: http://localhost:5050/#/experiments/10
Завершен шаг 1/25 | Лучший NDCG: 0.9814


[I 2026-05-26 15:34:50,828] Trial 2 finished with value: 0.9810606098063186 and parameters: {'learning_rate': 0.011430983876313222, 'depth': 10, 'l2_leaf_reg': 6.41003510568888}. Best is trial 0 with value: 0.9813587808725716.


🏃 View run trial_2 at: http://localhost:5050/#/experiments/10/runs/a8c6d05d44da42d0ad4dfa1237e4d449
🧪 View experiment at: http://localhost:5050/#/experiments/10
Завершен шаг 2/25 | Лучший NDCG: 0.9814

Лучшие параметры найдены: {'learning_rate': 0.023688639503640783, 'depth': 10, 'l2_leaf_reg': 7.587945476302646}
Обучаем финальную модель...
Считаю финальные метрики на тесте...


2026/05/26 16:11:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


---
Best mean NDCG@15: 0.1679
Best Precision@15: 0.0777
Best Recall@15: 0.2506
🏃 View run CatBoost_Optuna_Best at: http://localhost:5050/#/experiments/10/runs/27a76f567ec945079ef39cfae1b7418f
🧪 View experiment at: http://localhost:5050/#/experiments/10


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/alex/VS_python_projects/DreamTeamRecSys/.venv313/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
    ~~~~~~~~^^
  File "/home/alex/VS_python_projects/DreamTeamRecSys/.venv313/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 348, in dispatch_control
    await self.process_control(msg)
  File "/home/alex/VS_python_projects/DreamTeamRecSys/.venv313/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 354, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/home/alex/VS_python_projects/DreamTeamRecSys/.venv313/lib/python3.13/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callb

In [29]:
svd_best = {
    "model":          "SVD (best)",
    "target":         "log_target",
    "top_k":          15,
    "ndcg":           0.135736,
    "precision":      0.026741,
    "recall":         0.026168,
    "hyperopt":       "grid (ручной)",
}

CatBoost_best = {
    "model":          "CatBoost (best, Optuna)",
    "target":         "log_target",
    "top_k":          15,
    "ndcg":           0.1679,
    "precision":      0.0777,
    "recall":         0.2506,
    "hyperopt":       "Optuna TPE, 25 trials, 5% days",
}

final_comparison = pl.DataFrame([svd_best, CatBoost_best])

ndcg_delta    = CatBoost_best["ndcg"]      - svd_best["ndcg"]
prec_delta    = CatBoost_best["precision"] - svd_best["precision"]
recall_delta  = CatBoost_best["recall"]    - svd_best["recall"]

print(f"NDCG@15:    SVD={svd_best['ndcg']:.6f}  CatBoost_best={CatBoost_best['ndcg']:.6f}  Δ={ndcg_delta:+.6f} ({ndcg_delta/svd_best['ndcg']*100:+.1f}%)")
print(f"Precision:  SVD={svd_best['precision']:.6f}  CatBoost_best={CatBoost_best['precision']:.6f}  Δ={prec_delta:+.6f}")
print(f"Recall:     SVD={svd_best['recall']:.6f}  CatBoost_best={CatBoost_best['recall']:.6f}  Δ={recall_delta:+.6f}")

final_comparison

NDCG@15:    SVD=0.135736  CatBoost_best=0.167900  Δ=+0.032164 (+23.7%)
Precision:  SVD=0.026741  CatBoost_best=0.077700  Δ=+0.050959
Recall:     SVD=0.026168  CatBoost_best=0.250600  Δ=+0.224432


model,target,top_k,ndcg,precision,recall,hyperopt
str,str,i64,f64,f64,f64,str
"""SVD (best)""","""log_target""",15,0.135736,0.026741,0.026168,"""grid (ручной)"""
"""CatBoost (best, Optuna)""","""log_target""",15,0.1679,0.0777,0.2506,"""Optuna TPE, 25 trials, 5% days"""


Итоговый результат после преминения Optuna на тесте из 5000 популярных товаров по всем пользователям оказался значительно лучше бейзлайна. Подбор гиперпараметров сработал корректно.

# Лучшие метрики:
NDCG@15: 0.17

Precision@15: 0.08

Recall@15: 0.25

Лучшие параметры найдены: {'learning_rate': 0.023688639503640783, 'depth': 10, 'l2_leaf_reg': 7.587945476302646}